In [0]:
from datetime import datetime

# --------------------------------------------------
# Performance config
# --------------------------------------------------
spark.conf.set("spark.sql.shuffle.partitions", 4)

# --------------------------------------------------
# Widgets (Job Parameters)
# --------------------------------------------------
dbutils.widgets.text("customer_id", "")
dbutils.widgets.text("object_name", "")
dbutils.widgets.text("source_system", "salesforce")
dbutils.widgets.text("bucket_path", "")
dbutils.widgets.text("sf_catalog_table", "")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bucket_path = dbutils.widgets.get("bucket_path")
sf_catalog_table = dbutils.widgets.get("sf_catalog_table")

# --------------------------------------------------
# Validation
# --------------------------------------------------
if customer_id == "" or object_name == "" or bucket_path == "" or sf_catalog_table == "":
    raise Exception("Missing required job parameters")

# --------------------------------------------------
# Timestamp folder creation
# --------------------------------------------------
now = datetime.now()
yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")
historical_s3_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/historical/{yyyy}/{mm}/{dd}/{hh}"

print("--------------------------------------------------")
print(f"🚀 Starting Historical Load")
print(f"Customer      : {customer_id}")
print(f"Object        : {object_name}")
print(f"Source        : {source_system}")
print(f"Catalog Table : {sf_catalog_table}")
print(f"S3 Path       : {historical_s3_path}")
print("--------------------------------------------------")

try:
    # 1. Fetch Max SystemModstamp for Watermark
    max_ts_df = spark.sql(f"SELECT MAX(SystemModstamp) as max_ts FROM {sf_catalog_table}")
    max_ts = max_ts_df.first()[0]

    if max_ts is None:
        print("No records found in source table.")
        dbutils.notebook.exit("0")

    print(f"📍 Max SystemModstamp found: {max_ts}")

    # 2. Read and Write to S3 (Single Data Scan)
    df_raw = spark.table(sf_catalog_table)
    
    (
        df_raw
        .repartition(4)
        .write
        .mode("append")
        .format("parquet")
        .option("compression", "snappy")
        .save(historical_s3_path)
    )

    print(f"✅ FINISHED! Successfully landed raw records to S3")

    # 3. Initialize Watermark Table with 1-Minute Lookback
    print("Initializing watermark...")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
            source_system STRING,
            object_name STRING,
            last_processed_timestamp TIMESTAMP
        ) USING DELTA
    """)
    
    # Notice the `- INTERVAL 1 MINUTE` added here
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW new_watermarks AS
        SELECT source_system, object_name, last_processed_timestamp
        FROM ingestion_metadata.watermarks
        WHERE NOT (source_system = '{source_system}' AND object_name = '{object_name}')
        UNION ALL
        SELECT '{source_system}' AS source_system, '{object_name}' AS object_name, TIMESTAMP('{max_ts}') - INTERVAL 1 MINUTE AS last_processed_timestamp
    """)
    
    spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")
    print(f"✅ Watermark initialized successfully (minus 1 minute).")

    dbutils.notebook.exit("Success")

except Exception as e:
    print(f"❌ Pipeline Failed: {str(e)}")
    raise e